Re-ordering this script so it comes first before any preprocessing.

**NOTES:**
- The 'missing_TRO_and_ESP' column marks fMRI files that are missing TRO (TotalReadoutTime) and ESP (EchoSpacing) parameters; this is a "non-fatal" issue that can cause distortion correction and physical-space validation (e.g. SDC-SyN) to be less accurate; referencing this column can be (optionally) used to skip these processing phases, if desired.
- The 'EXPORT_WARNING_CSV' toggle parameter controls whether the list of such files is exported as a separate 'fMRI_params_incomplete.csv' file (but the 'missing_TRO_and_ESP' column is always stored in the main fMRI parameter index file regardless).

- We round voxel sizes ('vox_x', 'vox_y', 'vox_z') to the nearest thousandth decimal place for index readability (these are likely spurious floating-point rounding jitters anyways); however the original file metadata fields themselves remain un-altered, so downstream processes (e.g. nilearn, nibabel, FreeSurfer) will still read the original, precise values directly from the input files.


**TO CHECK / NOTES TO SELF:**
- **Not sure what happens with the "clustering" code if the input fields don't exist; should probably be toggle-able...**
-------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path
import subprocess

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, json, re, glob
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib

In [ ]:
### SET PARAMETERS, AND INPUT & OUTPUT FILEPATHS:

### PARAMETERS:
HARD_STOP = config['hard_errors']

EXPORT_WARNING_CSV = config['export_fMRI_metadata_warnings']

DEFAULT_DROP_FIRST_N = config['default_n_drop_frames']

### INPUTS:
ROOT_DIR = Path(config['root_output_directory'])
fMRI_DATA_DIR = Path(config['fMRI_data_directory'])
DATA_INDEX_PATH = Path(ROOT_DIR) / 'master_data_catalogue.csv'
DATA_INDEX = pd.read_csv(DATA_INDEX_PATH)

### OUTPUTS:
PARAMETER_INDEX_FILEPATH = Path(ROOT_DIR) / 'fMRI_parameter_index.csv'

print(f"Main project directory set as: {ROOT_DIR}")
print(f"Main fMRI root directory set as: {fMRI_DATA_DIR}")
print(f"Master data catalogue set as: {DATA_INDEX_PATH}")
print()
print(f"fMRI parameter catalogue will be saved to: {PARAMETER_INDEX_FILEPATH}")

First, let's build a base index of files to grab parameters for, by taking all non-NaN values from our various 'filename' columns, along with subject-, group-, and session IDs:

In [ ]:
# =========================
# Build base fMRI index from DATA_INDEX
# =========================

# Identify all columns containing both 'fMRI' and 'filename':
fmri_filename_cols = [c for c in DATA_INDEX.columns if ('fMRI' in c) and ('filename' in c)]

records = []
for _, row in DATA_INDEX.iterrows():
    subject_id = row.get('subject_ID', np.nan)
    group_id   = row.get('group_ID',   np.nan)
    for col in fmri_filename_cols:
        val = row[col]
        # keep only non-empty values:
        if pd.notna(val) and str(val).strip():
            # Derive session_ID from the column name: 'fMRI_<session>_filename':
            parts = col.split('_')
            session_id = parts[1] if len(parts) >= 3 else np.nan
            records.append({
                'subject_ID':     subject_id,
                'group_ID':       group_id,
                'session_ID':     session_id,
                'fMRI_filename':  str(val).strip()})  # <-- base path (no filetype extension)

# Assemble the dataframe:
index = pd.DataFrame.from_records(
    records,
    columns=['subject_ID', 'group_ID', 'session_ID', 'fMRI_filename'])

# Ensure uniqueness by filename (keep first occurrence):
if not index.empty:
    before = len(index)
    index = index.drop_duplicates(subset=['fMRI_filename']).reset_index(drop=True)
    removed = before - len(index)
else:
    removed = 0

print(f"[INFO] Scanned {len(fmri_filename_cols)} fMRI filename columns.")
print(f"[INFO] Collected {len(records)} total filename hits.")
print(f"[INFO] Removed {removed} duplicate filename rows.")
print(f"[OK] Built base index with {len(index)} unique fMRI files.")

Next, we'll recursively search for each individual 'fMRI_file' within the specified fMRI_ROOT_DIR structure, and explicitly check for matching .nii (or nii.gz) and .json file pairs; any mismatches will trigger error behavior based on the 'HARD_STOP' setting (itself a call to the 'hard_errors' field from config.yaml):

In [ ]:
# =========================
# Recursively resolve pairs for each fMRI basename under fMRI_DATA_DIR
# - Exact filename match (basename only), anywhere under root
# - JSON must be in the SAME directory as the chosen NIfTI
# - Store absolute base path (no extension) + chosen NIfTI filetype
# - HARD_STOP controls raise vs print+drop on mismatches
# =========================

# Initialize output columns (fill then optionally drop bad rows):
index = index.copy()
index['fMRI_basepath'] = np.nan  # absolute path, no extension
index['nii_filetype']  = np.nan  # '.nii' or '.nii.gz'

mismatch_rows = []

for i, row in index.iterrows():
    base_str = str(row['fMRI_filename']).strip()
    basename = Path(base_str).name  # <-- matches by basename anywhere under root directory

    # Find exact matches anywhere under fMRI_DATA_DIR:
    nii_list   = list(Path(fMRI_DATA_DIR).rglob(f"{basename}.nii"))
    niigz_list = list(Path(fMRI_DATA_DIR).rglob(f"{basename}.nii.gz"))
    json_list  = list(Path(fMRI_DATA_DIR).rglob(f"{basename}.json"))

    n_nii, n_niigz, n_json = len(nii_list), len(niigz_list), len(json_list)
    total_nii = n_nii + n_niigz

    chosen_nifti = None
    chosen_ext = None

    if total_nii == 1:
        if n_niigz == 1:
            chosen_nifti = niigz_list[0]
            chosen_ext = '.nii.gz'
        else:
            chosen_nifti = nii_list[0]
            chosen_ext = '.nii'

    ok = False
    if chosen_nifti is not None:
        # JSON must be in same dir with same basename:
        if chosen_ext == '.nii.gz':
            expected_json = chosen_nifti.with_name(chosen_nifti.name[:-7] + '.json')
        else:
            expected_json = chosen_nifti.with_suffix('.json')
        ok = expected_json.exists()

    if ok:
        # Strip NIfTI suffix to store basepath (no extension):
        if chosen_ext == '.nii.gz':
            base_noext = chosen_nifti.with_name(chosen_nifti.name[:-7])
        else:
            base_noext = chosen_nifti.with_name(chosen_nifti.name[:-4])
        index.at[i, 'fMRI_basepath'] = base_noext.resolve().as_posix()
        index.at[i, 'nii_filetype']  = chosen_ext
    else:
        issues = []
        if total_nii == 0:
            issues.append("no NIfTI (.nii/.nii.gz) found under root")
        elif total_nii > 1:
            issues.append(f"multiple NIfTIs found under root ({total_nii}); expected exactly one")
        if chosen_nifti is not None:
            if chosen_ext == '.nii.gz':
                expected_json = chosen_nifti.with_name(chosen_nifti.name[:-7] + '.json')
            else:
                expected_json = chosen_nifti.with_suffix('.json')
            if not expected_json.exists():
                issues.append("JSON sidecar missing in same folder as chosen NIfTI")

        mismatch_rows.append({
            'subject_ID':   row.get('subject_ID', np.nan),
            'group_ID':     row.get('group_ID',   np.nan),
            'session_ID':   row.get('session_ID', np.nan),
            'fMRI_filename': base_str,
            'basename':     basename,
            'n_.nii':       n_nii,
            'n_.nii.gz':    n_niigz,
            'n_.json':      n_json,
            'issue':        "; ".join(issues) if issues else "pairing failed"})

# Determine mismatches (i.e. anything that failed to populate both columns):
mismatch_mask = index['fMRI_basepath'].isna() | index['nii_filetype'].isna()
n_mismatch = int(mismatch_mask.sum())

if n_mismatch > 0:
    print(f"[WARN] {n_mismatch} entries failed strict pairing (expected exactly one NIfTI and one JSON in the same folder).")
    for rec in mismatch_rows[:25]:  # print up to 25 examples
        print(
            f"  - sub={rec['subject_ID']} grp={rec['group_ID']} ses={rec['session_ID']} "
            f"basename='{rec['basename']}' from '{rec['fMRI_filename']}' "
            f"NIfTI: .nii={rec['n_.nii']} .nii.gz={rec['n_.nii.gz']}; JSON={rec['n_.json']} :: {rec['issue']}")

    if HARD_STOP:
        raise RuntimeError("Hard fail due to missing/ambiguous fMRI file pairs. See warnings above.")
    else:
        kept_before = len(index)
        index = index.loc[~mismatch_mask].reset_index(drop=True)
        print(f"[INFO] HARD_STOP is False -> dropped {n_mismatch} mismatching rows; kept {len(index)} of {kept_before}.")
else:
    print("[OK] All entries have exactly one NIfTI and one JSON in the same folder.")

Next, we harvest our first set of parameters from the NIfTI file headers:

In [ ]:
# =========================
# NIfTI header extraction
# Adds: n_x, n_y, n_z, n_t, vox_x, vox_y, vox_z, TR_from_header, orientation_codes
# Also stores rounded and canonicalized versions of voxel dimensions for clean summaries.
# =========================

# Initialize columns:
index['n_x'] = np.nan
index['n_y'] = np.nan
index['n_z'] = np.nan
index['n_t'] = np.nan
index['vox_x'] = np.nan
index['vox_y'] = np.nan
index['vox_z'] = np.nan
index['TR_from_header'] = np.nan
index['orientation_codes'] = ""

errors = []

for i, row in index.iterrows():
    nifti_path = f"{row['fMRI_basepath']}{row['nii_filetype']}"
    try:
        img = nib.load(nifti_path)
        shape = list(img.shape)
        while len(shape) < 4:
            shape.append(1)
        n_x, n_y, n_z, n_t = shape[:4]

        zooms = img.header.get_zooms()
        vx = zooms[0] if len(zooms) > 0 else np.nan
        vy = zooms[1] if len(zooms) > 1 else np.nan
        vz = zooms[2] if len(zooms) > 2 else np.nan
        TR_header = float(zooms[3]) if len(zooms) > 3 else np.nan

        orient = "".join(nib.aff2axcodes(img.affine))

        index.at[i, 'n_x'] = n_x
        index.at[i, 'n_y'] = n_y
        index.at[i, 'n_z'] = n_z
        index.at[i, 'n_t'] = n_t
        index.at[i, 'vox_x'] = vx
        index.at[i, 'vox_y'] = vy
        index.at[i, 'vox_z'] = vz
        index.at[i, 'TR_from_header'] = TR_header
        index.at[i, 'orientation_codes'] = orient
    except Exception as e:
        errors.append((row.get('subject_ID', np.nan), row.get('session_ID', np.nan), nifti_path, str(e)))

if errors:
    print(f"[WARN] {len(errors)} NIfTI header(s) could not be read.")
    for sub, ses, p, msg in errors[:20]:
        print(f"  - sub={sub} ses={ses} path='{p}': {msg}")
    if HARD_STOP:
        raise RuntimeError("Hard fail due to unreadable NIfTI headers.")
else:
    print("[OK] Read NIfTI headers for all rows.")

### _______________________
### Rounding and canonicalization of voxel sizes:

# Round voxel sizes to 3 decimals for summary / QC:
index['vox_x'] = pd.to_numeric(index['vox_x'], errors='coerce').round(3)
index['vox_y'] = pd.to_numeric(index['vox_y'], errors='coerce').round(3)
index['vox_z'] = pd.to_numeric(index['vox_z'], errors='coerce').round(3)

print("\n[INFO] Rounded voxel sizes to 3 decimals for catalog readability.")
print("[INFO] Canonicalized vox_z (4.0±0.01 → 4.0 ; 5.0±0.01 → 5.0).")

Next we pull the next set of parameters, this time coming from the .json sidecar files:

In [ ]:
# =========================
# JSON sidecar extraction — also auto-drops empty dummy-scan columns
# Keeps original catalog names; captures alternates; removes unused dummy columns if all-NaN.
# =========================

index['TR_from_json'] = np.nan
index['EchoTime'] = np.nan
index['TotalReadoutTime'] = np.nan
index['EffectiveEchoSpacing'] = np.nan
index['SliceTiming_len'] = np.nan
index['SliceTiming_unique_count'] = np.nan

# Tries multiple possible variations on field names for dummy scan frames; drops them if nothing is found:
index['NumberOfVolumesDiscardedByScanner'] = np.nan
index['NumberOfVolumesDiscardedByUser'] = np.nan
index['NonSteadyStateVolumes_count'] = np.nan       # aux, may drop
index['NonSteadyStateVolumes_last_index'] = np.nan  # aux, may drop

index['Manufacturer'] = np.nan
index['ManufacturersModelName'] = np.nan

index['PED_raw_BIDS'] = np.nan
index['PhaseEncodingAxis_raw'] = np.nan
index['InPlanePhaseEncodingDirectionDICOM'] = np.nan

errors = []

for i, row in index.iterrows():
    json_path = f"{row['fMRI_basepath']}.json"
    try:
        with open(json_path, "r") as f:
            js = json.load(f)
    except Exception as e:
        errors.append((row.get('subject_ID', np.nan), row.get('session_ID', np.nan), json_path, str(e)))
        continue

    # Core timings & distortion parameters:
    index.at[i, 'TR_from_json'] = js.get("RepetitionTime", np.nan)
    index.at[i, 'EchoTime'] = js.get("EchoTime", np.nan)
    index.at[i, 'TotalReadoutTime'] = js.get("TotalReadoutTime", np.nan)
    index.at[i, 'EffectiveEchoSpacing'] = js.get("EffectiveEchoSpacing", np.nan)

    # Slice-timing summary:
    st_list = js.get("SliceTiming", None)
    if isinstance(st_list, list):
        index.at[i, 'SliceTiming_len'] = len(st_list)
        try:
            arr = np.array(st_list, dtype=float)
            uniq = np.unique(np.round(arr, 5))
            index.at[i, 'SliceTiming_unique_count'] = int(len(uniq))
        except Exception:
            index.at[i, 'SliceTiming_unique_count'] = np.nan

    # Discard/dummy volumes:
    lower_map = {k.lower(): k for k in js.keys()}

    if 'numberofvolumesdiscardedbyuser' in lower_map:
        k = lower_map['numberofvolumesdiscardedbyuser']
        index.at[i, 'NumberOfVolumesDiscardedByUser'] = pd.to_numeric(js.get(k), errors='coerce')
    elif 'dummy_scans' in lower_map:
        k = lower_map['dummy_scans']
        index.at[i, 'NumberOfVolumesDiscardedByUser'] = pd.to_numeric(js.get(k), errors='coerce')
    elif 'dummy_trs' in lower_map:
        k = lower_map['dummy_trs']
        index.at[i, 'NumberOfVolumesDiscardedByUser'] = pd.to_numeric(js.get(k), errors='coerce')

    # Scanner-provided count:
    if 'numberofvolumesdiscardedbyscanner' in lower_map:
        k = lower_map['numberofvolumesdiscardedbyscanner']
        index.at[i, 'NumberOfVolumesDiscardedByScanner'] = pd.to_numeric(js.get(k), errors='coerce')

    # NonSteadyStateVolumes (list of indices) — keep summaries only:
    if 'nonsteadystatevolumes' in lower_map:
        k = lower_map['nonsteadystatevolumes']
        nss = js.get(k)
        if isinstance(nss, list) and len(nss) > 0:
            try:
                nss_int = [int(x) for x in nss]
                index.at[i, 'NonSteadyStateVolumes_count'] = len(nss_int)
                index.at[i, 'NonSteadyStateVolumes_last_index'] = max(nss_int)
            except Exception:
                pass

    # Manufacturer / model:
    index.at[i, 'Manufacturer'] = js.get("Manufacturer", np.nan)
    index.at[i, 'ManufacturersModelName'] = js.get("ManufacturersModelName", np.nan)

    # PED-related raw fields:
    ped_dir = js.get("PhaseEncodingDirection", None)  # 'i','i-','j','j-'
    index.at[i, 'PED_raw_BIDS'] = ped_dir if isinstance(ped_dir, str) else np.nan
    phase_axis = js.get("PhaseEncodingAxis", None)    # 'i','j','k'
    index.at[i, 'PhaseEncodingAxis_raw'] = phase_axis if isinstance(phase_axis, str) else np.nan
    inplane = js.get("InPlanePhaseEncodingDirectionDICOM", None)  # 'ROW'/'COL'
    index.at[i, 'InPlanePhaseEncodingDirectionDICOM'] = inplane if isinstance(inplane, str) else np.nan

if errors:
    print(f"[WARN] {len(errors)} JSON sidecar(s) could not be read or parsed.")
    for sub, ses, p, msg in errors[:20]:
        print(f"  - sub={sub} ses={ses} path='{p}': {msg}")
    if HARD_STOP:
        raise RuntimeError("Hard fail due to unreadable JSON sidecars.")
else:
    print("[OK] Parsed JSON sidecars for all rows (robust).")

# Auto-drop unused dummy columns if they are entirely NaN:
maybe_drop_cols = [
    'NumberOfVolumesDiscardedByScanner',
    'NumberOfVolumesDiscardedByUser',
    'NonSteadyStateVolumes_count',
    'NonSteadyStateVolumes_last_index']
to_drop = [c for c in maybe_drop_cols if c in index.columns and index[c].isna().all()]

if to_drop:
    index = index.drop(columns=to_drop)
    print(f"[INFO] Dropped unused dummy-related columns (all NaN): {to_drop}")
else:
    print("[INFO] Kept dummy-related columns (at least one non-NaN value present).")


Next, we add a batch of "derived" metadata/parameters:

In [ ]:
# =========================
# Derived fields
#
# Policy for extracting / inferring 'drop_first_n':
#   1) Use User value if numeric (>=0)
#   2) else Scanner value if numeric (>=0)
#   3) else NonSteadyStateVolumes -> (max index + 1)
#   4) else default_drop_first_n (config, fallback 5)
# =========================

index['drop_first_n'] = np.nan
index['PED_axis'] = np.nan
index['PED_axis_source'] = np.nan
index['PED_axis_is_inferred'] = np.nan
index['PED_sign'] = np.nan

for i, row in index.iterrows():
    # get 'drop_first_n':
    v_user = pd.to_numeric(row.get('NumberOfVolumesDiscardedByUser'), errors='coerce')
    v_scan = pd.to_numeric(row.get('NumberOfVolumesDiscardedByScanner'), errors='coerce')
    v_nss_count = pd.to_numeric(row.get('NonSteadyStateVolumes_count'), errors='coerce')
    v_nss_last  = pd.to_numeric(row.get('NonSteadyStateVolumes_last_index'), errors='coerce')

    chosen = np.nan
    if pd.notna(v_user) and v_user >= 0:
        chosen = int(v_user)
    elif pd.notna(v_scan) and v_scan >= 0:
        chosen = int(v_scan)
    elif pd.notna(v_nss_last):
        # If list exists, be conservative & drop through the last listed non-steady index:
        chosen = int(v_nss_last) + 1
    elif pd.notna(v_nss_count):
        # Fallback: if only 'count' is available, use that:
        chosen = int(v_nss_count)
    else:
        chosen = DEFAULT_DROP_FIRST_N

    index.at[i, 'drop_first_n'] = int(chosen)

    # PED sign derivation:
    ped_raw = row.get('PED_raw_BIDS')
    phase_axis = row.get('PhaseEncodingAxis_raw')
    dicom_inplane = row.get('InPlanePhaseEncodingDirectionDICOM')

    ped_valid = isinstance(ped_raw, str) and re.match(r'^[ijk](?:-)?$', ped_raw) is not None
    if ped_valid:
        axis_from_bids = ped_raw[0]
    else:
        axis_from_bids = np.nan

    axis_from_phase = phase_axis if isinstance(phase_axis, str) and phase_axis in ('i','j','k') else np.nan

    if isinstance(dicom_inplane, str):
        v = dicom_inplane.strip().upper()
        axis_from_dicom = "j" if v == "ROW" else ("i" if v == "COL" else np.nan)
    else:
        axis_from_dicom = np.nan

    if isinstance(axis_from_bids, str):
        index.at[i, 'PED_axis'] = axis_from_bids
        index.at[i, 'PED_axis_source'] = "json_phaseencodingdirection"
        index.at[i, 'PED_axis_is_inferred'] = False
    elif isinstance(axis_from_phase, str):
        index.at[i, 'PED_axis'] = axis_from_phase
        index.at[i, 'PED_axis_source'] = "json_phaseencodingaxis"
        index.at[i, 'PED_axis_is_inferred'] = True
    elif isinstance(axis_from_dicom, str):
        index.at[i, 'PED_axis'] = axis_from_dicom
        index.at[i, 'PED_axis_source'] = "dicom_inplane"
        index.at[i, 'PED_axis_is_inferred'] = True
    else:
        index.at[i, 'PED_axis'] = np.nan
        index.at[i, 'PED_axis_source'] = "missing"
        index.at[i, 'PED_axis_is_inferred'] = np.nan

    if ped_valid:
        index.at[i, 'PED_sign'] = '-' if ped_raw.endswith('-') else '+'
    else:
        index.at[i, 'PED_sign'] = 'unknown'

# Final sanity-check:
if index['PED_axis'].isna().any():
    bad = index.loc[index['PED_axis'].isna(), ['fMRI_basepath','PED_raw_BIDS','PhaseEncodingAxis_raw','InPlanePhaseEncodingDirectionDICOM']]
    print("[WARN] Some runs have no PED axis after priority-based extraction. Examples:")
    print(bad.head(15).to_string(index=False))
    if HARD_STOP:
        raise RuntimeError("Hard fail: missing PED axis for some runs.")
else:
    print("[OK] PED axis present for all runs (priority-based).")

Dropping redundant PED-parameter columns (all info we need is now fully covered by 'PED_axis', 'PED_sign' and 'PED_axis_source'):

In [ ]:
# Drop redundant PED-parameter fields:
drop_raw = [c for c in ['PED_raw_BIDS','PhaseEncodingAxis_raw','InPlanePhaseEncodingDirectionDICOM']
            if c in index.columns]
if drop_raw:
    index = index.drop(columns=drop_raw)
    print(f"[INFO] Dropped raw PED inputs: {drop_raw}")

Post-QC normalization + reports:

- One main thing we do here is add a 'missing_TRO_and_ESP' column, which marks files lacking these parameters (e.g. so that SDC-SyN can be skipped for these files, or other "special-treatment" processing forks, can be applied later downstream, if desired)
  - My understanding is that the actual signal-processing effects of these missing parameters are typically very minor at most, so I don't actually plan on implementing this kind of contextual forking in the pipeline at this point.

In [ ]:
# =========================
# Post-QC normalization + non-catalog outputs (with TR diagnostics + warning export toggle)
# =========================

# Set 1 ms tolerance:
TR_TOL_S = 0.001

# Hard check -- 'TR_from_json' is present everywhere:
tr_series = pd.to_numeric(index['TR_from_json'], errors='coerce')
if tr_series.isna().any():
    missing_rows = index.loc[
        tr_series.isna(),
        ['subject_ID', 'group_ID', 'session_ID', 'fMRI_basepath']]
    print("[ERR] Some TR_from_json values are missing:")
    print(missing_rows.head(20).to_string(index=False))
    raise RuntimeError("Hard fail: missing TR_from_json values.")

# Verify exactly one unique TR across all files (triggers diagnostics if not):
tr_round6 = tr_series.round(6)
unique_tr = np.sort(tr_round6.dropna().unique())

if len(unique_tr) != 1:
    vc = tr_round6.value_counts().sort_index()
    majority_tr = vc.idxmax()
    deviation_mask = tr_round6.ne(majority_tr)
    n_dev = int(deviation_mask.sum())

    print(f"[ERR] TR_from_json is not uniform across dataset (rounded to 6 decimals).")
    print(f"      Majority TR: {majority_tr} s; Deviations: {n_dev} row(s).")

    if n_dev <= 10:
        cols_show = [
            'subject_ID', 'group_ID', 'session_ID',
            'fMRI_basepath', 'TR_from_json']
        print("[DETAIL] Deviating rows (up to 10):")
        print(index.loc[deviation_mask, cols_show].to_string(index=False))
    else:
        print("[DETAIL] TR_from_json value_counts (rounded to 6 decimals):")
        print(vc.to_string())

    if HARD_STOP:
        raise RuntimeError(
            "Hard fail: TR_from_json is not uniform across dataset.")
    else:
        print("[WARN] HARD_STOP is False → proceeding despite multiple TRs.")
else:
    print(f"[OK] TR_from_json uniform across all files: {unique_tr[0]} s")

# Ensure header-vs-JSON TR agreement (within tolerance):
tr_hdr = pd.to_numeric(index['TR_from_header'], errors='coerce')
tr_json = tr_series
diff = (tr_hdr - tr_json).abs()
n_bad = int((diff > TR_TOL_S).sum())

if n_bad > 0:
    examples = index.loc[
        diff > TR_TOL_S,
        ["subject_ID", "group_ID", "session_ID", "fMRI_basepath", "TR_from_header", "TR_from_json"]].head(12)

    print("[ERR] TR_from_header not within ±1 ms of TR_from_json for some runs. Examples:")
    print(examples.to_string(index=False))
    raise RuntimeError(
        f"Hard fail: {n_bad} run(s) have TR_from_header not within ±1 ms of TR_from_json.")
else:
    print("[OK] TR_from_header matches TR_from_json within ±1 ms for all runs.")

# Normalization -- keeps JSON value:
index = index.drop(columns=["TR_from_header"]).rename(
    columns={"TR_from_json": "RepetitionTime"})
print("[INFO] Standardized: kept JSON value as 'RepetitionTime'; dropped 'TR_from_header'.")

# Readout timing availability (TRO/ESP) + per-row flag:
tro = pd.to_numeric(index["TotalReadoutTime"], errors="coerce")
esp = pd.to_numeric(index["EffectiveEchoSpacing"], errors="coerce")
missing_mask = tro.isna() & esp.isna()

# Add boolean flag to main table:
index['missing_TRO_and_ESP'] = missing_mask
n_missing_rt = int(missing_mask.sum())

if n_missing_rt > 0:
    print(f"[WARN] {n_missing_rt} run(s) lack both TotalReadoutTime and EffectiveEchoSpacing.")
else:
    print("[OK] Readout timing present in all runs (TRO and/or ESP).")

# Print summaries:
print("\n[INFO] PED_sign distribution ('+','-','unknown'):")
print(index["PED_sign"].value_counts(dropna=False).to_string())

print("\n[INFO] PED_axis counts (i/j/k):")
print(index["PED_axis"].value_counts(dropna=False).to_string())

Final step: Adding covariance cluster labels:

These columns help identify and control for **non-biological variation** in the fMRI data that arises from differences in scanners or acquisition settings, rather than from subjects’ brain activity:
- ~~'site_code'; a simple label that distinguishes file groups only by research site (i.e. the first 3 characters of 'subject_ID's)~~ (Removed because this is not a universal feature of BIDS datasets)
- 'protocol_code'; a more fine-grained label that distinguishes file "clusters" based on the machine/model used, **and** any variations among the other fMRI scanning parameters used for different subjects within a given research site (as it appears, at least in this dataset, that significant parameter/protocol differences were made across subjects, even within the same scanning site)

Such site-level differences can introduce systematic offsets in signal intensity, noise patterns, or timing parameters; and so these two columns can be used as labels for adding 'site_code' or 'protocol_code' as a covariates (nuisance regressors) in GLMs, regressions, or classifiers to partial out site/scanner effects.

- "...These can then be included as covariates or batch factors in any statistical model, machine-learning regressor, or ComBat-style harmonization procedure." [...] "This step is often called *harmonization metadata labeling*, and it’s commonly used in multi-site fMRI analyses before statistical modeling or batch correction."


Note that we do "filter out" minute differences in parameters (which typically arise due to trivial rounding errors / tiny "floating-point jitters" / etc.) by setting the "tolerance parameters" included in the code cell -- I don't think users will need to fiddle with these so I won't add them to config.yaml at this point; but in any case they are here if they need to be adjusted / made user-facing later.

In [ ]:
# =========================
# Scanner / protocol clustering with tolerance-based canonicalization
# (tolerance knobs defined here for easy later promotion to config.yaml)
# =========================

# TOLERANCE KNOBS (can be moved to config.yaml later):
REP_TIME_DECIMALS = 3       # RepetitionTime rounding in seconds (ms precision)
ECHO_TIME_DECIMALS = 3      # EchoTime rounding in seconds (ms precision)
VOX_Z_DECIMALS   = 2        # vox_z rounding in mm (0.01 mm)
TRO_STEP         = 0.0005   # TotalReadoutTime quantization step in seconds (0.5 ms)
ESP_STEP         = 0.000005 # EffectiveEchoSpacing quantization step in seconds (5 µs)

# Fields that characterize scanner/sequence configuration:
scanner_keys = [
    "Manufacturer",
    "ManufacturersModelName",
    "RepetitionTime",
    "EchoTime",
    "vox_z",
    "TotalReadoutTime",
    "EffectiveEchoSpacing"]
scanner_keys = [c for c in scanner_keys if c in index.columns]

# Canonicalized view with parameter-specific tolerances:
canon = index[scanner_keys].copy()

if "RepetitionTime" in canon.columns:
    canon["RepetitionTime"] = (
        pd.to_numeric(canon["RepetitionTime"], errors="coerce")
        .round(REP_TIME_DECIMALS))

if "EchoTime" in canon.columns:
    canon["EchoTime"] = (
        pd.to_numeric(canon["EchoTime"], errors="coerce")
        .round(ECHO_TIME_DECIMALS))

if "vox_z" in canon.columns:
    canon["vox_z"] = (
        pd.to_numeric(canon["vox_z"], errors="coerce")
        .round(VOX_Z_DECIMALS))

if "TotalReadoutTime" in canon.columns:
    tro = pd.to_numeric(canon["TotalReadoutTime"], errors="coerce")
    canon["TotalReadoutTime"] = (np.round(tro / TRO_STEP) * TRO_STEP).round(6)

if "EffectiveEchoSpacing" in canon.columns:
    esp = pd.to_numeric(canon["EffectiveEchoSpacing"], errors="coerce")
    canon["EffectiveEchoSpacing"] = (np.round(esp / ESP_STEP) * ESP_STEP).round(6)

# Build the clustering signature from canonicalized values:
canon_filled = canon.fillna("NA").astype(str)
index["scanner_signature"] = canon_filled.apply(
    lambda r: "|".join(r.values.tolist()),
    axis=1)

# Assign compact alphabetic labels (A, B, ..., Z, AA, AB, ...):
unique_sigs = index["scanner_signature"].drop_duplicates().reset_index(drop=True)
n = len(unique_sigs)

labels = []
for i in range(n):
    x = i
    s = ""
    while True:
        s = chr(65 + (x % 26)) + s
        x = x // 26 - 1
        if x < 0:
            break
    labels.append(s)

sig_to_label = dict(zip(unique_sigs, labels))
index["protocol_code"] = index["scanner_signature"].map(sig_to_label)

# Canonical summary per cluster (so tiny raw diffs don’t duplicate rows):
cluster_summary = canon.copy()
cluster_summary["protocol_code"] = index["protocol_code"]
cluster_summary = (
    cluster_summary
    .drop_duplicates()
    .sort_values(["protocol_code"])[["protocol_code"] + scanner_keys])

print(f"[INFO] Assigned {len(sig_to_label)} unique 'protocol_code' labels (tolerance-aware).")
print(cluster_summary.to_string(index=False))

# Clean up temporary signature column:
index = index.drop(columns=["scanner_signature"])

---------
#### Final save / export:

Includes optional export of 'fMRI_params_incomplete.csv', if 'EXPORT_WARNING_CSV' toggle is enabled:

In [ ]:
# Optional export of warning rows (governed by EXPORT_WARNING_CSV global toggle):
if 'EXPORT_WARNING_CSV' in globals() and EXPORT_WARNING_CSV:
    problems_df = index.loc[missing_mask,
                            ["subject_ID","group_ID","session_ID",
                             "protocol_code", #'site_code",
                             "missing_TRO_and_ESP", "fMRI_basepath"]].copy()
    out_path = Path(ROOT_DIR) / "fMRI_params_incomplete.csv"
    problems_df.to_csv(out_path, index=False)
    print(f"\n[SAVED] Incomplete metadata CSV: {out_path} (rows={len(problems_df)})")
else:
    print("\n[INFO] EXPORT_WARNING_CSV is False → not writing fMRI_params_incomplete.csv")

index.to_csv(PARAMETER_INDEX_FILEPATH, index=False)
print(f"[SAVED] Main fMRI parameter catalog:  {PARAMETER_INDEX_FILEPATH}")